## Rag from Scratch 

### Environment 

In [ ]:
! pip install langchain langchain_community langchain_google_genai langchain-text-splitters chromadb bs4 

In [ ]:
import os 
os.environ['LANGCHAIN_TRACING_V2'] = 'false'
os.environ['LANGCHAIN_ENDPOINT'] = 'https://api.smith.langchain.com'
os.environ['LANGCHAIN_API_KEY'] = "<your_api_key>"
os.environ['GOOGLE_API_KEY'] = "<your_api_key>"

### Packages 

In [ ]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import Chroma
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.output_parsers import StrOutputParser
import bs4

### Indexing 

In [ ]:
# load data
loader = WebBaseLoader(
    web_paths=("https://lilianweng.github.io/posts/2023-06-23-agent/",),
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            class_=("post-content", "post-title", "post-header")
        )
    ),
)
blog_docs = loader.load()

In [ ]:
# splitting 
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=300, 
    chunk_overlap=50)

splits = text_splitter.split_documents(blog_docs)

### Retrieval

In [ ]:
# define vectorstore and then retriever 
embd = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")

vectorstore = Chroma.from_documents(documents=splits, 
                                    embedding=embd)

# transforming vectorstore in retriever (number of output splitts : k = 1)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

In [ ]:
# get retrieval chuncks 
retrieval_chuncks = retriever.invoke("What is Task Decomposition?")

### Generation 

In [ ]:
# Prompt
prompt = ChatPromptTemplate.from_template("""
You are an assistant for question-answering tasks.

Use the following retrieved context to answer the question.

If the answer is not in the context, say you don't know.

Context:
{context}

Question:
{question}

Answer:
""")

In [ ]:
# LLM and chain 
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)

chain = prompt | llm | StrOutputParser()


In [ ]:
result = chain.invoke({"context": retrieval_chuncks, "question": "What is Task Decomposition?"})
print(result)